# code for topic modeling of episode annotations and recall transcripts

### imports

In [1]:
import os
import re
import pickle
import numpy as np
import pandas as pd
from collections import defaultdict
from os.path import join as opj
from hypertools.tools import format_data as fit_transform
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from num2words import num2words
from scipy.interpolate import interp1d

# stops hypertools from opening PyQt process
%matplotlib inline

In [2]:
# try to show progress bars for long-running cells
# to properly show progress bars, you'll need to install tqdm,
# as well as widgetsnbextension and ipywidgets to render the element
try:
    from tqdm import tqdm_notebook as tqdm
    tqdm_pbar = True
except ModuleNotFoundError:
    print('To enable progress bars, install tqdm module (`pip install tqdm`)')
    from IPython.display import clear_output
    tqdm_pbar = False

### set paths

In [3]:
data_dir = '../../data'
annot_dir = opj(data_dir, 'annotations')
transc_dir = opj(data_dir, 'transcriptions', 'manual')
ep_traj_dir = opj(data_dir, 'models', 'episodes', 'trajectories')
rec_traj_dir = opj(data_dir, 'models', 'recalls', 'trajectories')
pickle_dir = opj(data_dir, 'pickles')

### load data

In [4]:
# formatted annotations
annotations = {episode: pd.read_csv(opj(annot_dir, f'{episode}.csv')) 
               for episode in ['atlep1', 'atlep2', 'arrdev']}

# across-session subject IDs
id_maps = pd.read_pickle(opj(pickle_dir, 'id_maps.p'))

### text preprocessing parameters

In [5]:
# text preprocessing
stop_words = [word.replace("'", '') for word in stopwords.words('english')]
# add some stop words from scikit-learn's list
sklearn_stopwords = ['even', 'something', 'around', 'cannot', 'anyone', 
                     'may', 'else', 'afterwards', 'thus', 'whether', 
                     'much', 'since', 'ie', 'others', 'either', 
                     'neither', 'per', 'many', 'lot', 'eg', 'example', 
                     'however', 'anything', 'back', 'cant', 'anyway', 
                     'another', 'nothing', 'us', 'itd', 'yet']
# add frequently used filler words and contractions
extra_stopwords = ['like', 'whatever', 'super', 'thats', 'also', 'theyre', 
                   'theyve', 'lot', 'stuff']

stop_words += sklearn_stopwords + extra_stopwords
stop_words = set(stop_words)

# convert treebank pos tags to wordnet pos tags
# consider pronouns as nouns, deteriminers as adjectives
# default to noun for all other tags
pos_mappings = defaultdict(lambda: 'n')
for tb_tag, wn_tag in zip(['N', 'P', 'V', 'J', 'D', 'R'],
                          ['n', 'n', 'v', 'a', 'a', 'r']):
    pos_mappings[tb_tag] = wn_tag

In [6]:
# words/multiword phrases to be combined or 
# considered equivalent, based on a few criteria
substitutions = {
    # combine multiword phrases and names
    'paper boy': 'paperboy',
    'george-michael': 'georgemichael',
    'flo rida': 'floxrida',
    'low key': 'lowkey',
    'low-key': 'lowkey',
    'parking lot': 'parkinglot',
    'night club': 'nightclub',
    'ex girlfriend': 'exgirlfriend',
    'ex-girlfriend': 'exgirlfriend',
    'ex wife': 'exwife',
    'ex-wife': 'exwife',
    # characters' nicknames are equivalent to full names
    'earnest': 'earn',
    'vanessa': 'van',
    'david': 'dave',
    # lead character is sometimes referred to using the
    # well known actor's name or artist pseudonym
    'donald glover': 'earn',
    'glover': 'earn',
    'childish gambino': 'earn',
    'gambino': 'earn',
    # explatives -- **note below**
    'f-word': 'fuck',
    'n-word': 'nigga',
    'racial slur': 'nigga',
    'offensive term': 'nigga',
    # other words/phrases
    'déja': 'deja',
    ' cause ': ' because ',
    ' weed ': ' marijuana ',
    ' pot ': ' marijuana ',
    # accepted english shortenings of words
    'gonna': 'going to',
    'wanna': 'want to',
    'kinda': 'kind of',
    'sorta': 'sort of',
    # common slight inaccuracies in characters' names
    'ernie': 'earn',
    'albert': 'alfred',
    'swift': 'swiff',
    'smith': 'swiff',
    'lonnie': 'lottie',
    'lan': 'van',
    'JP': 'KP',
    'darren': 'darius',
    'darrell': 'darius',
    'joe': 'gob',
    'mae': 'maeby',
    'lily': 'lindsay',
    'lilly': 'lindsay'
}

# As a series, Atlanta addresses numerous social issues through the lens of
# its characters' struggle to survive and find success in the face of poverty 
# and modern, systemic, institutionalized racism. Many of its scenes are 
# emotionally jarring and are intended to make viewers uncomfortable. One such
# commentary in the pilot episode involves a white charaacter, Dave, casually 
# using a racial slur in front of Earn, but not in front of other African
# American characters. 
# In recounting the episodes, participants naturally avoiding repeating such
# offensive language themselves, though they clearly remembered its presence. 
# Because our approach to characterizing memory entails mapping between 
# the explicit contents of experiences and verbal recalls, and because our
# annotations of the stimuli reflect their verbatim transcripts, we chose to
# substitute this offensive language for instances of euphamisms that clearly
# refer to its use.
# Racism exists in many forms and on many scales in the 21st century. For
# a perspective on the history and usage of the n-word, see 
# https://www.tolerance.org/magazine/fall-2011/straight-talk-about-the-nword

### topic modeling parameters

In [7]:
n_topics = 100
episode_wsize = 50    # annotations
recall_wsize = 200    # words

# vectorizer parameters
vectorizer_params = {
    'model' : 'CountVectorizer', 
    'params' : {
        'stop_words' : None    # stop words handled separately
    }
}

# topic model parameters
semantic_params = {
    'model' : 'LatentDirichletAllocation', 
    'params' : {
        'n_components' : n_topics,
        'learning_method' : 'batch',
        'random_state' : 0,
    }
}

# timestamp of last video frame
# used for interpolating timeseries
endframe_times = {
    'atlep1': 1466.0,
    'atlep2': 1316.52,
    'arrdev': 1236.6
}

## functions

In [8]:
# lower_nopunc = [re.sub("[^\w\s-]+", '', chunk.lower()) for chunk in episode_bag]
# no_acc = [x.replace('é', 'e') for x in lower_nopunc]
# no_digit = [re.sub(r"(\d+)", lambda x: num2words(int(x.group(0))), chunk) for chunk in no_acc]
# spaced = [' '.join(x.replace(',', ' ').split()) for x in no_digit]

### for text preprocessing and document formatting

In [9]:
def format_text(text):
    text = ' '.join(list(text.dropna()))
    punc_stripped = re.sub("[^\w\s-]+", '', text.lower())
    no_digit = re.sub(r"(\d+)", lambda x: num2words(int(x.group(0))), punc_stripped)
    spaced = ' '.join(no_digit.replace(',', ' ').split())
    return spaced

In [10]:
def remove_stopwords(text):
    textlist = text.split()
    sw_removed = ' '.join([word for word in textlist if word not in stop_words])
    return sw_removed

In [11]:
lemmatizer = WordNetLemmatizer()

def lemmatize(text, pos_dict=pos_mappings):
    words_tags = pos_tag(text.split())
    lemmas = []
    for word, tag in words_tags:
        lemma = lemmatizer.lemmatize(word, pos_dict[tag[0]])
        lemmas.append(lemma)
    return ' '.join(lemmas)

In [12]:
def preprocess_text(data, data_type=None):
    if data_type == 'episode':
        df = data.loc[:, 'Narrative details (external events)':'Setting']
    elif data_type == 'recall':
        df = pd.DataFrame(np.atleast_2d(data))
    else:
        raise ValueError("Episode vs recall data not specified")
        
    words_bag = df.apply(lambda x: format_text(x), axis=1)
    # combine multiword tokens, standardize names, deal with euphemisms, etc.
    replaced = words_bag.replace(substitutions, regex=True)
    # remove remaining hyphens
    no_hyphen = replaced.replace('-', ' ', regex=True)
    # remove stop words
    no_stopwords = no_hyphen.apply(lambda x: remove_stopwords(x))
    lemmatized = no_stopwords.apply(lambda x: lemmatize(x))
    preprocessed = lemmatized.tolist()
    return preprocessed[0].split() if data_type == 'recall' else preprocessed

In [13]:
# def create_windows(textlist, wsize, taper_beginning=False, taper_end=True):
#     windows = []
#     if taper_beginning:
#         # first `wsize` windows start with first item and grow until 
#         # full window size. Ensures first `wsize` items and last `wsize` 
#         # items are in equal number of windows if taper_end is True
#         for ix in range(1, wsize):
#             windows.append(' '.join(textlist[0 : ix]))
#     if taper_end:
#         # continue appending windows through last item, though last 
#         # `wsize` windows will contain < `wsize` items
#         n_indices = len(textlist)
#     else:
#         # stop shifting sliding window when < `wsize` items remain
#         n_indices = len(textlist) - (wsize - 1)
        
#     for ix in range(n_indices):
#         windows.append(' '.join(textlist[ix : ix + wsize]))
#     return windows

In [14]:
def create_windows(textlist, wsize):
    windows = []
    for ix in range(wsize // 2, wsize):
        windows.append(' '.join(textlist[0 : ix]))
    
    for ix in range(len(textlist) - wsize // 2 + 1):
        windows.append(' '.join(textlist[ix : ix + wsize]))
        
    return windows

### for interpolating episode trajectories

In [15]:
# def find_midpoint_times(df, endframe_time):
#     """
#     returns list of timepoints at middle of each annotation segment
#     """
#     midpoint_times = []
#     for i, tpt in enumerate(df['Onset time']):
#         try:
#             midpoint_times.append((tpt + df['Onset time'][i+1]) / 2)
#         except KeyError:
#             # use final frame's timestamp
#             midpoint_times.append((tpt + endframe_time) / 2)
#     return midpoint_times

In [16]:
def find_midpoint_times(df, endframe_time, wsize):
    onsets = df['Onset time']
    midpoint_times = []
    
    for ix in range(wsize // 2, wsize):
        midpoint_times.append(onsets[ix] / 2)
        
    for ix in range(len(onsets) - wsize // 2 + 1):
        try:
            midpoint_times.append((onsets[ix] + onsets[ix + wsize]) / 2)
        except KeyError:
            midpoint_times.append((onsets[ix] + endframe_time) / 2)
        
    return midpoint_times

In [17]:
def get_recall_xvals(rec_windows, wsize, new_xmax):
    xvals = [0]
    next_xval = 0
    
    for window in rec_windows[1:]:
        delta_x = len(window.split()) / wsize
        next_xval += delta_x
        xvals.append(next_xval)
        
    scaled_xvals = np.array(xvals) * (new_xmax / xvals[-1])
    return scaled_xvals

In [18]:
def interpolate_trajectory(traj, new_xmax, documents=None, windows=None, resolution=1):
    if isinstance(documents, pd.DataFrame):
        # get middle timepoint for each window
        curr_xvals = find_midpoint_times(documents, new_xmax, episode_wsize)
    else:
        # scale number of recall windows to episode length
        curr_xvals = get_recall_xvals(windows, recall_wsize, new_xmax)
        
    new_timepoints = np.arange(round(new_xmax), step=resolution)
    interp_func = interp1d(curr_xvals, traj, axis=0, fill_value='extrapolate')
    return interp_func(new_timepoints)

## main topic modeling function

In [19]:
def transform_text(documents, vec_params=vectorizer_params, sem_params=semantic_params, 
                   corpus=None, interp_len=None, return_windows=False):
    if isinstance(documents, pd.DataFrame):
        data_type = 'episode'
        window_size = episode_wsize
    elif isinstance(documents, str):
        data_type = 'recall'
        window_size = recall_wsize
        if not corpus:
            raise ValueError("You must pass a training corpus to transform recall transcripts")
            
    processed_docs = preprocess_text(documents, data_type=data_type)
    windows = create_windows(processed_docs, window_size)
    corpus = windows if data_type == 'episode' and not corpus else corpus
    traj = fit_transform(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
    if interp_len:
        traj = interpolate_trajectory(traj, interp_len, documents, windows)
        
    return (traj, windows) if return_windows else traj

In [20]:
# atlep1_df = annotations['atlep1']
# with open('../../data/transcriptions/manual/MD-021919-A-02/debugQzo2F:debugV7e7L/debugQzo2F:debugV7e7L-recall.txt',
#          'r') as f:
#     a1_test = f.read()

In [21]:
# def fit_and_transform(documents, vec_params=vectorizer_params, sem_params=semantic_params, 
#                         corpus=None, resample_shape=None, return_windows=False):
#     # handle annotations
#     if isinstance(documents, pd.DataFrame):
#         windows = create_windows(documents, data_type='episode')
#         corpus = windows if not corpus else corpus
#         # fit topic model and transform documents
#         traj =  hyp.tools.format_data(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
        
#         # interpolate to length of episode (seconds)
#         if return_windows:
#             return interpolate_episode(traj, documents), windows
#         else:
#             return interpolate_episode(traj, documents)
        
#     # handle recall transcripts
#     elif isinstance(documents, str):
#         if not corpus:
#             raise ValueError("You must pass a training corpus to transform recall transcripts")
#         windows = get_recall_windows(documents, recall_wsize)
#         traj =  hyp.tools.format_data(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
        
#         # resample to corresponding episode length
#         return resample(traj, resample_shape) 

## model episode content, get sliding windows for fitting recall models

In [22]:
episode_trajs = dict.fromkeys(annotations.keys())
recall_corpora = dict.fromkeys(annotations.keys())

for episode, annotations_df in annotations.items():
    interp_len = endframe_times[episode]
    traj, windows = transform_text(annotations_df, 
                                   interp_len=interp_len, 
                                   return_windows=True)
    episode_trajs[episode] = traj
    recall_corpora[episode] = windows

## save episode trajectories

In [23]:
# for episode, traj in episode_trajs.items():
#     np.save(opj(ep_traj_dir, f'{episode}_trajectory'), traj)

## load in and model recall transcripts

In [24]:
# # try to show progress bars for long-running cells
# # to properly show progress bars, you'll need to install tqdm,
# # as well as widgetsnbextension and ipywidgets to render the element
# try:
#     from tqdm import tqdm_notebook as tqdm
#     def iter_transcripts(top, **kwargs):
#         total = 0
#         for root, dirs, files in os.walk(top, **kwargs):
#             dirs[:] = [d for d in dirs if d != 'drops']
#             files[:] = [f for f in files if f.startswith('debug')]
#             for file in files:
#                 with open(opj(root, file), 'r') as f:
#                     total += len(f.read().split())

#         with tqdm(total=total, unit='transcripts', leave=False) as pbar:
#             for root, dirs, files in os.walk(top, **kwargs):
#                 dirs[:] = [d for d in dirs if d != 'drops']
#                 files[:] = [f for f in files if f.startswith('debug')]
#                 yield root, dirs, files
                
#                 for file in files:
#                     pass
    
# except ModuleNotFoundError:
#     print('To enable progress bars, install tqdm module (`pip install tqdm`)')
#     def walkdir()

In [25]:
# total_files = 0
# n_subjects = 57
# for root, dirs, files in os.walk(transc_dir):
#     dirs[:] = [d for d in dirs if d != 'drops']
#     files[:] = [f for f in files if f.startswith('debug')]
#     total_files += len(files)
#     for file in files:
#         # base progress on transcript length for cleaner estimate
#         with open(opj(root, file), 'r') as f:
#             total_files += len(f.read().split())

n_subjects = id_maps.shape[0]
total_files = n_subjects * 3

In [33]:
# if tqdm_pbar:
#     pbar = tqdm(total=total_files, unit='transcripts', leave=False)
# else:
#     pct, frac, curr = 0, 0, 0
#     bar, pad = '', ' '*80
#     pbar = f"{pct}% [{bar}{pad}] {frac} {curr}"

In [ ]:
recall_trajectories = {rectype: {} for rectype in ['atlep1', 
                                                   'delayed',
                                                   'atlep2',
                                                   'arrdev']}

if tqdm_pbar:
    itersubjects = tqdm(id_maps.iterrows(), total=total_files, leave=False)
else:
    itersubjects = id_maps.iterrows()
        
for sid, turkids in itersubjects:
    for ses in [1, 2]:
        turkid = turkids[f'session {ses}']
        if ses == 1:
            rectypes = ['atlep1']
        elif 'A' in sid:
            rectypes = ['delayed', 'atlep2']
        else:
            rectypes = ['delayed', 'arrdev']
            
        for rectype in rectypes:
            if rectype == 'delayed':
                ext = 'delayed'
                corpus = recall_corpora['atlep1']
                interp_len = endframe_times['atlep1']
            else:
                ext = 'recall'
                corpus = recall_corpora[rectype]
                interp_len = endframe_times[rectype]
                
            fpath = opj(transc_dir, sid, turkid, f'{turkid}-{ext}.txt')
            try:
                with open(fpath, 'r') as f:
                    itersubjects.update(1)
                    transcript = f.read()
                    
                if not transcript:
                    continue
                    
                traj = transform_text(transcript, corpus=corpus, interp_len=interp_len)
                recall_trajectories[rectype][turkid] = traj
                if tqdm_pbar:
                    itersubjects.update(1)
                    
            except FileNotFoundError:
                itersubjects.update(1)
                continue

In [162]:
# x = {'ep': episode_trajs, 'rec': recall_trajectories}
# with open('/Users/paxtonfitzpatrick/Desktop/tmptrajs.p', 'wb') as f:
#     pickle.dump(x, f)

In [17]:
# with open('/Users/paxtonfitzpatrick/Desktop/tmptrajs.p', 'rb') as f:
#     x = pickle.load(f)
# episode_trajs = x['ep']
# recall_trajectories = x['rec']

In [23]:
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
# fig, axarr = plt.subplots(nrows=7, ncols=5)
# fig.set_size_inches(25,20)
# axarr = axarr.flatten()
# ax_idx = 0
# for sid, row in id_maps.iterrows():
#     tid1 = row['session 1']
#     tid2 = row['session 2']
#     try:
#         immtraj = recall_trajectories['atlep1'][tid1]
#         deltraj = recall_trajectories['delayed'][tid2]
#         hyp.plot([immtraj, deltraj], ndims=2, reduce='ppca', ax=axarr[ax_idx], show=False)
#         ax_idx += 1
#     except KeyError:
#         pass

# axarr[ax_idx].axis('off')
# fig.subplots_adjust(hspace=.1, wspace=.1)
# # fig.savefig('/Users/paxtonfitzpatrick/Desktop/new_imm_del_umap.pdf')
# display(fig)

In [ ]:
# recall_trajectories = {
#     'atlep1' : [],
#     'prediction' : [],
#     'delayed' : [],
#     'atlep2' : [],
#     'arrdev' : []
# }

# total = sum([len([f for f in files if f.endswith('corrected.wav.txt')]) for r, d, files in os.walk(transc_dir)])
# # walk transcription directory structure
# currfile = 1
# for root, dirs, files in os.walk(transc_dir):
    
#     # FOR USE WITH AUTOMATIC TRANSCRIPTS -- REMOVE WHEN SWITCHING TO MANUAL
#     transcripts = [f for f in files if f.endswith('corrected.wav.txt')]
#     for transc in transcripts:

#         # assign correct episode windows, corresponding episode trajectory shape, and dict key
#         if any('prediction' in t for t in transcripts) or 'delayed' in transc:
#             corpus = atlep1_windows
#             resample_shape = atlep1_traj.shape[0]
#             if 'recall' in transc:
#                 rectype = 'atlep1'
#             elif 'prediction' in transc:
#                 rectype = 'prediction'
#             elif 'delayed' in transc:
#                 rectype = 'delayed'
#             else:
#                 raise ValueError('Transcript is not a recognized option')
            
#         elif '-A-' in root:
#             corpus = atlep2_windows
#             resample_shape = atlep2_traj.shape[0]
#             rectype = 'atlep2'
            
#         else:
#             corpus = arrdev_windows
#             resample_shape = arrdev_traj.shape[0]
#             rectype = 'arrdev'
            
#         with open(opj(root ,transc), 'r') as f:
#             # FOR USE WITH AUTOMATIC TRANSCRIPTS -- REMOVE WHEN SWITCHING TO MANUAL
#             transcript = ' '.join([line.split(',')[0].lower() for line in f.read().split('\n')])
            
#         # fit topic model to episode annotations, 
#         print(f'modeling transcript {currfile}/{total}...    {transc}')
#         p_traj = fit_and_transform(transcript, resample_shape=resample_shape, corpus=corpus)
        
#         recall_trajectories[rectype].append((transc.split('-')[0],p_traj))
#         currfile += 1

## save individual trajectories

In [ ]:
# for rectype, data in recall_trajectories.items():
#     for (turkid, traj) in data:
#         np.save(opj(rec_traj_dir, rectype, f'{turkid}.npy'), traj)

## create and save average recall trajectories

In [ ]:
# for rectype, data in recall_trajectories.items():
#     avg_trajectory = np.array([traj for (turkid, traj) in data]).mean(axis=0)
#     np.save(opj(rec_traj_dir, rectype, 'avg_trajectory.npy'), avg_trajectory)